In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 4
learning_rate = 3e-4
max_iters = 1000
eval_iters = 250

cuda


In [2]:
with open ('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
#print(chars)
vocab_size = len(chars)
#print(vocabulary_size)


In [3]:
string_to_int = { ch:i for i, ch in enumerate(chars) }
int_to_string = { i:ch for i, ch in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype = torch.long)
print(data[:100])
print(len(data))

tensor([80,  1, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32, 29,  1,
        47, 33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1, 26, 49,
         0,  0,  1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45, 37,  0,
         0,  1,  1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,  1, 47,
        33, 50, 25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,  1, 36,
        25, 38, 28,  1, 39, 30,  1, 39, 50,  9])
232326


In [4]:
n = int(0.8 * len(data))
train_data = data[:n]
val_data = data[n:]

x = train_data[:block_size]
y = train_data[:block_size+1]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size, ))
    print(ix)
    x = torch.stack([data [i:i+ block_size] for i in ix])
    y = torch.stack([data [i+1: i+block_size + 1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('inputs')
print(x)
print('targets:')
print(y)

tensor([ 88383, 143271,  18347,  29607])
inputs
tensor([[ 1, 54, 65, 73, 61, 68, 74, 60],
        [74, 65, 57,  1, 60, 68,  1, 63],
        [57,  0, 69, 54, 72, 72, 58, 57],
        [58, 57,  1, 73, 61, 62, 72,  1]], device='cuda:0')
targets:
tensor([[54, 65, 73, 61, 68, 74, 60, 61],
        [65, 57,  1, 60, 68,  1, 63, 74],
        [ 0, 69, 54, 72, 72, 58, 57,  1],
        [57,  1, 73, 61, 62, 72,  1, 69]], device='cuda:0')


In [5]:
@torch.no_grad
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [6]:
class BigramLanguageModel(nn.Module):
        def __init__(self, vocab_size):
            super().__init__()
            self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

        def forward(self, index, targets = None):
            logits = self.token_embedding_table(index)

            if targets is None:
                loss = None
            else:
                B, T, C = logits.shape
                logits = logits.view(B*T, C)
                targets = targets.view(B*T)
                loss = F.cross_entropy(logits, targets)

            return logits, loss

        def generate(self, index, max_new_tokens):
            for _ in range(max_new_tokens):
                logits, loss = self.forward(index)
                logits = logits[:, -1, :]
                probs = F.softmax(logits, dim=-1)
                index_next = torch.multinomial(probs, num_samples = 1)
                index = torch.cat((index, index_next), dim = -1)
            return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

In [7]:
optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss:{losses['train']}, val loss: {losses['val']}")
    xb, yb = get_batch('train')

    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()
print(loss.item())

tensor([126224,  40279, 144045,  37581])
tensor([143892,  16259, 152034, 164829])
tensor([  1072,  26034, 115716, 127866])
tensor([134952, 176219, 146383, 146600])
tensor([72814, 55240, 40643, 88514])
tensor([178753, 174104,  79164, 138056])
tensor([152230,  35896,  94290,  41996])
tensor([ 93179, 172464,  21688,  92157])
tensor([ 31035, 160586, 133511, 182763])
tensor([172416,  20636, 146639,  17705])
tensor([150526,  24755,  85137,   9889])
tensor([117255, 108753,  23471,  50309])
tensor([ 26990,   3543, 135785,  79413])
tensor([149374,  90052,  58651,  73163])
tensor([57631, 29076, 39256, 30067])
tensor([104353,  54249, 133536,  76962])
tensor([152385, 120927,  33603,  50836])
tensor([128436,  65085,   3194,  88410])
tensor([148749, 174408,  57608, 173833])
tensor([ 44635, 101642,  38927, 169764])
tensor([123798, 142808, 139573, 146142])
tensor([150994, 131485,  47281, 175124])
tensor([123522, 134790,  69343, 126993])
tensor([ 57133, 128550,  34238, 129943])
tensor([109624,  54931, 

In [8]:
context = torch.zeros((1, 1), dtype = torch.long, device = device)
generated_chars = decode(m.generate(context, max_new_tokens = 500)[0].tolist())
print(generated_chars)


a
UcZrHqemwSKT_DZJh3!"MevacX-8-VWtPZa2Nm)_,lGBRRirCxrpoFM(3j!U&FTcLYd9BW2,in80LPvd9h*RB[8?[76q.l0mGnb]Y,-D[CZ
&kdNLk?)nPc;hC)J.]X-7BZI2qc5G, 7qyRj5fV3rTsp&_",oQVWAV!p6E_O ]X-﻿FqNykX:qw0nak
XAWW9MV[c9WP﻿d5aftL.qz1')6w]C5ckp.wO;﻿3o[y,2JNy5a7.ez7fUFOE!?vV,CRT1jLd.!0f'b MZBxD&7d91jufKESsIBika:'j
*]enF8y[
Va,﻿vFY'o8GC!k4LQE8i1jT'tKE_B:3A5U?XqTB
RSu*!0:'PtxAin![y:fM(VpgjuV_"z1
e;X-&,E?odWc zURm8G,w]JY1zX4Ub
RnmQP7CGb2iftIGlG.qVr0T[y28,CV1]371]ys&JQWbVmlGn(YNhJ
"ck?in]qUPx1QXW?qqKE!lGmYJioO-JG;RrjHj.!l
